<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Extracción de datos de diferentes fuentes: Archivos de Excel</h3>
    </div>
</div>


# Archivos Excel

Excel puede contener varias hojas, fórmulas, índices y celdas de presentación. Al extraer datos, distingue la hoja que contiene la tabla de las filas de título o notas que sirven para lectura humana.

`pd.read_excel` es conveniente para leer directamente una hoja. `ExcelFile` resulta útil cuando se desea inspeccionar o reutilizar un libro con varias hojas sin abrirlo repetidamente.

> **Buenas prácticas:** registra el nombre de la hoja, comprueba los encabezados y convierte explícitamente fechas y números cuando Excel los haya interpretado de forma ambigua.


In [ ]:
# Configuración común para ejecutar esta sección de forma independiente
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dir_base = os.getcwd()
print(dir_base)
ruta = os.path.join(dir_base, 'Data')
print(ruta)


In [ ]:
# A partir de la función
pd.read_excel(ruta+'/API_SI.POV.DDAY_DS2_en_excel_v2_1930012.xls')

In [ ]:
#!pip install xlrd >= 2.0.1

In [ ]:
# Importamos la clase ExcelFile


In [ ]:
# A partir de la clase

# Importa la primera página

### Un caso real: el libro del Banco Mundial

El archivo que estamos leyendo proviene del **Banco Mundial** y contiene el indicador *"Poverty headcount ratio at \$1.90 a day"* (porcentaje de población en pobreza extrema). Es un ejemplo perfecto para entender por qué `ExcelFile` es tan práctico, porque el libro **no es una sola tabla**: está formado por varias hojas con propósitos distintos.

```mermaid
flowchart LR
    LIBRO[Libro .xls] --> D[Hoja 'Data'<br/>valores por país y año]
    LIBRO --> C[Hoja 'Metadata - Countries'<br/>región e ingreso]
    LIBRO --> I[Hoja 'Metadata - Indicators'<br/>definición del indicador]
```

La clase `ExcelFile` **abre el archivo una sola vez** y mantiene el libro en memoria. A partir de ese objeto podemos inspeccionar sus hojas, leer solo la que nos interese y aplicar parámetros distintos a cada una, sin volver a leer el disco cada vez.

In [ ]:
# sheet_names muestra todas las hojas del libro sin leer sus datos


Si leemos la hoja `Data` tal cual, las **tres primeras filas** contienen texto de presentación (*"Data Source"*, *"Last Updated Date"*…), no los datos. Por eso los encabezados reales (`Country Name`, `Country Code`, años…) están en la **cuarta fila**.

El argumento `header=3` le indica a pandas que use esa fila como encabezado y descarte las anteriores. Compara el resultado con la lectura ingenua para ver la diferencia.

In [ ]:
# Hoja 'Data' con el encabezado en la fila correcta (se descartan las 3 filas de presentación)


In [ ]:
# La hoja de metadatos de países sí tiene el encabezado en la primera fila


#### Leer todas las hojas de una sola vez

Con `sheet_name=None`, `pd.read_excel` devuelve un **diccionario** `{nombre_de_hoja: DataFrame}`. Es la forma más cómoda de cargar un libro completo y recorrer sus hojas de forma programática.

In [ ]:
# sheet_name=None carga el libro completo como un diccionario de DataFrames
libro = pd.read_excel(ruta + '/API_SI.POV.DDAY_DS2_en_excel_v2_1930012.xls', sheet_name=None)

#Mostrar las hojas con el tamaño


#### Combinar hojas

El valor real aparece al **unir** las hojas. La hoja `Data` dice *cuánta* pobreza hay por país; la hoja `Metadata - Countries` dice a qué **región** y **grupo de ingreso** pertenece cada país. Uniéndolas por `Country Code` podemos responder preguntas que ninguna hoja contesta por sí sola.

In [ ]:
# Unimos la región y el grupo de ingreso de cada país a la tabla de datos
df_enriquecido = df_data.merge(
    df_paises[['Country Code', 'Region', 'IncomeGroup']],
    on='Country Code',
    how='left'
)
df_enriquecido[['Country Name', 'Country Code', 'Region', 'IncomeGroup']].head()

In [ ]:
# ¿Cuántos países hay por grupo de ingreso? (una pregunta que ninguna hoja responde sola)


> **¿`pd.read_excel` o `ExcelFile`?**
> - Usa **`pd.read_excel`** cuando solo necesitas **una hoja** de forma rápida.
> - Usa **`ExcelFile`** cuando vas a leer **varias hojas del mismo libro**: abre el archivo una sola vez y luego llamas a `.parse()` tantas veces como quieras, lo que es más eficiente.
>
> **Motor de lectura:** los archivos `.xls` (antiguos) requieren `xlrd`, mientras que los `.xlsx` modernos usan `openpyxl`. Si aparece un error de motor, instala el paquete correspondiente con `%pip install xlrd` u `openpyxl`.

### Práctica de Laboratorio: Extracción de datos Archivos Excel

**Título: Análisis del indicador de pobreza del Banco Mundial**

1. Abrir el archivo con `ExcelFile`.
2. Listar sus hojas.
3. Leer correctamente la hoja `Data` usando `header=3`.
4. Leer la hoja `Metadata - Countries`.
5. Integrar ambas hojas mediante `Country Code`.
6. Transformar los años a formato largo con [`melt()`](https://pandas.pydata.org/docs/reference/api/pandas.melt.html).
7. Calcular promedios por país, año, región y grupo de ingreso.
8. Crear dos gráficas con `matplotlib`.
9. Exportar los resultados a un nuevo archivo Excel.


In [ ]:
#importar librerías
from pandas import ExcelFile

#Cargar datos

# 1. Abrir el archivo con ExcelFile (se abre una sola vez)

# 2. Listar las hojas del libro

# 3. Leer la hoja 'Data' con header=3 (se descartan las 3 filas de presentación)

# 4. Leer la hoja 'Metadata - Countries'


In [ ]:
# 5. Integrar Data con los metadatos de país (región y grupo de ingreso)


In [ ]:
# 6. Pasar los años (columnas) a formato largo con melt(). Las columnas de años (`1960.0`, `1961.0`, …) 
# se convierten en dos columnas: `Año` y `Pobreza`. El formato largo facilita agrupar y graficar.


In [ ]:
# 7. Promedios por cada dimensión


In [ ]:
# 8a. Pobreza promedio por región (barras horizontales)


In [ ]:
# 8b. Evolución del promedio mundial a lo largo del tiempo


In [ ]:
# 9. Exportar los resultados a un nuevo archivo Excel con varias hojas
